# MSigDB decoupleR GSEA Saturation Analysis (study subsampling)

**Environment:** `clamp-analyses`

For each CLAMP model in the study-level saturation grid (`08_saturation_study`, varying K across study coverage levels and up to 3 seeds), this notebook:

1. Loads the Z matrix (gene loadings per LV).
2. Filters genes to the model gene universe overlapping MSigDB (v2026.1).
3. Runs `decoupleR::run_fgsea()` on the full loading matrix (all LVs as columns), using the raw LV loadings as the ranking statistic (no top-N filtering, no absolute value).
4. Keeps only **positive-side** enrichment (`statistic == "norm_fgsea"` and `score > 0`) — i.e. MSigDB terms enriched among genes with high *positive* LV loadings. The negative tail is discarded entirely.
5. Stores raw `terms_padj`: the minimum BH-adjusted p-value per MSigDB term across all LVs (BH-adjustment done within each LV, then minimum taken across LVs).
6. Saves per-model RDS caches (`_msigdb_decoupler_gsea.rds`) and per-pct-level summary RDS/CSV. FDR thresholds are applied in `01_msigdb_decoupler_gsea_plot.ipynb`.

In [ ]:
library(here)
library(dplyr)
library(decoupleR)

## Paths

In [ ]:
models_dir <- here("output/01_model_building/04_archs4/08_saturation_study")
output_dir <- here("output/03_model_biology/00_archs4/08_saturation_study/decoupler_gsea")

dir.create(file.path(output_dir, "CLAMPfull"), recursive = TRUE, showWarnings = FALSE)
dir.create(file.path(output_dir, "CLAMPbase"), recursive = TRUE, showWarnings = FALSE)

## Model grid: discovered dynamically from disk

Scans `08_saturation_study` for all `study_saturation_rs{pct}_k{k}_seed_{seed}` directories that have a `CLAMPfull_hall/Z.csv`. K values and study coverage levels are not hardcoded.

In [ ]:
all_subdirs <- list.dirs(models_dir, recursive = FALSE, full.names = FALSE)

model_grid <- do.call(rbind, lapply(all_subdirs, function(d) {
  m <- regmatches(d, regexec("^study_saturation_rs([0-9]+)_k([0-9]+)_seed_([0-9]+)$", d))[[1]]
  if (length(m) < 4) return(NULL)
  z_path_full <- file.path(models_dir, d, "CLAMPfull_hall", "Z.csv")
  if (!file.exists(z_path_full)) return(NULL)
  data.frame(
    rs_pct        = as.integer(m[2]),
    k_val         = as.integer(m[3]),
    seed          = as.integer(m[4]),
    subdir        = d,
    z_path_full   = z_path_full,
    z_path_base   = file.path(models_dir, d, "CLAMPbase", "Z.csv"),
    stringsAsFactors = FALSE
  )
}))

if (is.null(model_grid) || nrow(model_grid) == 0) {
  stop("No saturation models found in: ", models_dir)
}

model_grid <- model_grid[order(model_grid$rs_pct, model_grid$k_val, model_grid$seed), ]
rownames(model_grid) <- NULL

message("Available models: ", nrow(model_grid))
print(model_grid[, c("rs_pct", "k_val", "seed")])

## Load MSigDB gene sets as a decoupleR network

In [ ]:
msig_gmt <- clusterProfiler::read.gmt(here("data/pathways/msigdb.v2026.1.Hs.symbols.gmt"))
net <- msig_gmt %>% dplyr::rename(source = term, target = gene)
message(sprintf("MSigDB gene sets loaded: %d", length(unique(net$source))))

## Helper: get n_studies from subsample_info

`subsample_info.rds` lives in the upstream `07_bp_coverage_study` source dirs, not in the model dir itself.

In [ ]:
study_base_dir <- here("output/01_model_building/04_archs4/07_bp_coverage_study")
study_dirs     <- list.dirs(study_base_dir, recursive = FALSE, full.names = FALSE)

pct_to_subdir <- list()
for (d in study_dirs) {
  m <- regmatches(d, regexec("([0-9]+)$", d))[[1]]
  if (length(m) >= 2) pct_to_subdir[[as.character(as.integer(m[2]))]] <- d
}

get_n_studies <- function(rs_pct, seed_idx) {
  subdir <- pct_to_subdir[[as.character(rs_pct)]]
  if (is.null(subdir)) return(NA_integer_)
  src_path <- file.path(
    study_base_dir, subdir,
    sprintf("study_coverage_rs%d_seed_%d", rs_pct, seed_idx),
    "subsample_info.rds"
  )
  if (file.exists(src_path)) readRDS(src_path)$n_studies else NA_integer_
}

## Helper: run decoupleR GSEA for one model

Returns a list with raw `terms_padj` (minimum BH-adjusted p-value per MSigDB term across all LVs), positive-side only.

In [ ]:
run_gsea_for_model <- function(z_path, n_cores = 4) {
  Z              <- read.csv(z_path, row.names = 1, check.names = FALSE)
  universe_genes <- rownames(Z)
  mat            <- as.matrix(Z[universe_genes %in% net$target, , drop = FALSE])
  n_lvs          <- ncol(mat)

  term_overlap   <- tapply(net$target %in% rownames(mat), net$source, sum)
  n_total_msigdb <- sum(term_overlap >= 10L)

  gsea_res <- decoupleR::run_fgsea(mat = mat, network = net, minsize = 10, nproc = n_cores)

  # positive-side only: enrichment among genes with high positive LV loadings
  gsea_res <- gsea_res %>% dplyr::filter(statistic == "norm_fgsea", score > 0)

  # BH-adjust within each LV (condition), then take min adjusted p per pathway across LVs
  gsea_res <- gsea_res %>%
    dplyr::group_by(condition) %>%
    dplyr::mutate(p_adj = p.adjust(p_value, method = "BH")) %>%
    dplyr::ungroup()

  # n_samples via B.csv header only (avoids loading the full multi-GB model rds)
  b_path <- file.path(dirname(z_path), "B.csv")
  n_samples <- if (file.exists(b_path)) {
    ncol(read.csv(b_path, nrows = 0, check.names = FALSE))
  } else NA_integer_

  list(
    n_samples      = n_samples,
    n_lvs          = n_lvs,
    n_total_msigdb = n_total_msigdb,
    terms_padj     = tapply(gsea_res$p_adj, gsea_res$source, min)
  )
}

## Helper: build results row from ORA output

In [ ]:
build_row <- function(spec, res, n_studies) {
  data.frame(
    rs_pct           = spec$rs_pct,
    k_val            = spec$k_val,
    seed             = spec$seed,
    n_studies        = n_studies,
    n_samples        = res$n_samples,
    n_lvs            = res$n_lvs,
    n_total_msigdb   = res$n_total_msigdb,
    stringsAsFactors = FALSE
  )
}

## Run GSEA: CLAMPfull

In [ ]:
run_model_grid <- function(model_subdir, z_col) {
  results_list <- lapply(seq_len(nrow(model_grid)), function(i) {
    spec      <- model_grid[i, ]
    z_path    <- spec[[z_col]]
    n_studies <- get_n_studies(spec$rs_pct, spec$seed)

    cache_path <- file.path(
      output_dir, model_subdir,
      sprintf("rs%d_k%d_seed%d_msigdb_decoupler_gsea.rds", spec$rs_pct, spec$k_val, spec$seed)
    )

    if (!file.exists(z_path)) { warning("Z.csv not found: ", z_path); return(NULL) }

    if (file.exists(cache_path)) {
      message(sprintf("Loading cached: %s rs%d k%d seed%d", model_subdir, spec$rs_pct, spec$k_val, spec$seed))
      res <- readRDS(cache_path)
      if (is.null(res$n_studies)) res$n_studies <- n_studies
    } else {
      message(sprintf("Running decoupleR GSEA: %s rs%d k%d seed%d", model_subdir, spec$rs_pct, spec$k_val, spec$seed))
      res <- run_gsea_for_model(z_path)
      if (!is.null(res)) {
        res$n_studies <- n_studies
        saveRDS(res, cache_path)
      }
    }

    if (is.null(res)) return(NULL)
    build_row(spec, res, n_studies)
  })

  results_df <- do.call(rbind, Filter(Negate(is.null), results_list))
  rownames(results_df) <- NULL
  results_df %>% dplyr::arrange(rs_pct, k_val, seed)
}

results_full_df <- run_model_grid("CLAMPfull", "z_path_full")
message("Collected ", nrow(results_full_df), " CLAMPfull rows")
print(results_full_df)

In [ ]:
for (pct in sort(unique(results_full_df$rs_pct))) {
  pct_df   <- results_full_df[results_full_df$rs_pct == pct, ]
  rds_path <- file.path(output_dir, "CLAMPfull", sprintf("results_pct%d_msigdb_decoupler_gsea.rds", pct))
  csv_path <- file.path(output_dir, "CLAMPfull", sprintf("results_pct%d_msigdb_decoupler_gsea.csv", pct))
  saveRDS(pct_df, rds_path)
  write.csv(pct_df, csv_path, row.names = FALSE)
  message(sprintf("Saved CLAMPfull pct%d: %d models", pct, nrow(pct_df)))
}

## Run GSEA: CLAMPbase

In [ ]:
results_base_df <- run_model_grid("CLAMPbase", "z_path_base")
message("Collected ", nrow(results_base_df), " CLAMPbase rows")
print(results_base_df)

In [ ]:
for (pct in sort(unique(results_base_df$rs_pct))) {
  pct_df   <- results_base_df[results_base_df$rs_pct == pct, ]
  rds_path <- file.path(output_dir, "CLAMPbase", sprintf("results_pct%d_msigdb_decoupler_gsea.rds", pct))
  csv_path <- file.path(output_dir, "CLAMPbase", sprintf("results_pct%d_msigdb_decoupler_gsea.csv", pct))
  saveRDS(pct_df, rds_path)
  write.csv(pct_df, csv_path, row.names = FALSE)
  message(sprintf("Saved CLAMPbase pct%d: %d models", pct, nrow(pct_df)))
}